In [18]:
import subprocess, sys

print("Installing packages...")

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
    'langchain-core', 'langchain', 'langchain-community',
    'langchain-groq', 'pypdf'], capture_output=True)

# Verify
import importlib
for mod, name in [('langchain','LangChain'),('langchain_groq','LangChain-Groq'),('pypdf','PyPDF')]:
    try:
        m = importlib.import_module(mod)
        print(f"  [OK] {name}: {getattr(m,'__version__','installed')}")
    except:
        print(f"  [MISSING] {name}")

print("\nDone! Restart runtime now: Runtime > Restart Session > then continue from Block 2")

Installing packages...
  [OK] LangChain: 1.2.15
  [OK] LangChain-Groq: 1.1.2
  [OK] PyPDF: 6.9.2

Done! Restart runtime now: Runtime > Restart Session > then continue from Block 2


In [ ]:
!pip install langchain==0.2.16 langchain-core==0.2.41 langchain-community==0.2.16 langchain-groq


In [1]:
from groq import Groq
import os

# Set your API key
os.environ["GROQ_API_KEY"] = "xxxxxxxxxxxxxxxxxxxxxxxxxx"

client = Groq(api_key=os.environ["GROQ_API_KEY"])

# Fetch all available models
models = client.models.list()

# Print model names
print("Available Models:\n")
for model in models.data:
    print(model.id)

Available Models:

moonshotai/kimi-k2-instruct
groq/compound-mini
meta-llama/llama-prompt-guard-2-86m
whisper-large-v3-turbo
qwen/qwen3-32b
llama-3.1-8b-instant
openai/gpt-oss-20b
meta-llama/llama-prompt-guard-2-22m
allam-2-7b
openai/gpt-oss-120b
canopylabs/orpheus-arabic-saudi
meta-llama/llama-4-scout-17b-16e-instruct
canopylabs/orpheus-v1-english
whisper-large-v3
openai/gpt-oss-safeguard-20b
llama-3.3-70b-versatile
moonshotai/kimi-k2-instruct-0905
groq/compound


In [2]:
from langchain_groq import ChatGroq
import os

os.environ["GROQ_API_KEY"] = "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=0.1,
)

response = llm.invoke("Say hello in one sentence.")
print(response.content)

Hello, it's nice to meet you and I'm here to help with any questions or topics you'd like to discuss.


In [3]:
import os
os.makedirs('sample_claims', exist_ok=True)
os.makedirs('reports', exist_ok=True)

claim_1 = """
CHUBB INSURANCE - CLAIM DOCUMENT
==================================
Claim ID: CLM-2024-00123
Date of Filing: 2024-03-15
Policy Number: POL-789456

POLICYHOLDER INFORMATION
Name: Rajesh Kumar
Policy Type: Commercial Property Insurance
Coverage Amount: $500,000
Premium Status: Active
Policy Start Date: 2023-01-01
Policy End Date: 2025-01-01

INCIDENT DETAILS
Date of Incident: 2024-03-10
Type of Incident: Fire Damage
Location: Warehouse Unit 4B, Industrial Area, Bangalore
Description: Fire broke out due to electrical short circuit.
             60% of stored goods were damaged or destroyed.

DAMAGE ASSESSMENT
Property Damage: $120,000
Inventory Loss: $85,000
Business Interruption Loss: $30,000
Total Claimed Amount: $235,000

SUPPORTING DOCUMENTS
- Fire Department Report: Attached (FD-2024-0315)
- Police Report: Attached (PR-2024-0312)
- Photographs of Damage: 24 photos attached

WITNESS INFORMATION
Name: Suresh Mehta
Statement: Confirmed electrical fault as cause of fire

Signature: Rajesh Kumar, Date: 2024-03-15
"""

claim_2 = """
CHUBB INSURANCE - CLAIM DOCUMENT
==================================
Claim ID: CLM-2024-00456
Date of Filing: 2024-03-20
Policy Number: POL-112233

POLICYHOLDER INFORMATION
Name: Vikram Singh
Policy Type: Personal Property Insurance
Coverage Amount: $50,000
Premium Status: Active
Policy Start Date: 2024-02-01
Policy End Date: 2025-02-01

INCIDENT DETAILS
Date of Incident: 2024-03-18
Type of Incident: Theft
Location: Residential Apartment, MG Road, Bangalore
Description: Claimant reports theft of jewelry and electronics.
             No signs of forced entry reported by police.
             Neighbors report no unusual activity.

DAMAGE ASSESSMENT
Jewelry Loss: $28,000
Electronics Loss: $15,000
Cash Loss: $5,000
Total Claimed Amount: $48,000

SUPPORTING DOCUMENTS
- Police Report: Attached (PR-2024-0318)
- Photographs: 3 photos attached

WITNESS INFORMATION
None provided.

NOTES: Policy taken only 45 days ago. No forced entry.
Signature: Vikram Singh, Date: 2024-03-20
"""

claim_3 = """
CHUBB INSURANCE - CLAIM DOCUMENT
==================================
Claim ID: CLM-2024-00789
Date of Filing: 2024-04-02
Policy Number: POL-334455

POLICYHOLDER INFORMATION
Name: Priya Sharma
Policy Type: Commercial Property Insurance
Coverage Amount: $200,000
Premium Status: Active
Policy Start Date: 2022-06-01
Policy End Date: 2025-06-01

INCIDENT DETAILS
Date of Incident: 2024-03-28
Type of Incident: Water Damage
Location: Office Building, Koramangala, Bangalore
Description: Burst pipe caused flooding across 3 floors.
             Server room partially affected.

DAMAGE ASSESSMENT
Structural Damage: $60,000
Equipment Loss: $75,000
Data Recovery Costs: $15,000
Total Claimed Amount: $150,000

SUPPORTING DOCUMENTS
- Plumber Assessment Report: Attached
- Photographs of Damage: 38 photos attached

WITNESS INFORMATION
Name: Building Manager
Statement: Confirmed burst pipe on 3rd floor

Signature: Priya Sharma, Date: 2024-04-02
"""

with open('sample_claims/claim_001_fire.txt', 'w') as f: f.write(claim_1)
with open('sample_claims/claim_002_theft.txt', 'w') as f: f.write(claim_2)
with open('sample_claims/claim_003_water.txt', 'w') as f: f.write(claim_3)

print("Claims created:")
print("  claim_001_fire.txt   --> Expected: APPROVE")
print("  claim_002_theft.txt  --> Expected: ESCALATE")
print("  claim_003_water.txt  --> Expected: CONDITIONAL")

Claims created:
  claim_001_fire.txt   --> Expected: APPROVE
  claim_002_theft.txt  --> Expected: ESCALATE
  claim_003_water.txt  --> Expected: CONDITIONAL


In [4]:
import re, os
from datetime import datetime

# ── TOOL 1: Read Document ──────────────────────────────────────
def read_claim_document(file_path: str) -> str:
    file_path = file_path.strip().strip('"').strip("'")
    if file_path.endswith('.pdf'):
        try:
            from pypdf import PdfReader
            reader = PdfReader(file_path)
            return '[CLAIM DOC]\n' + ''.join(p.extract_text() for p in reader.pages)
        except Exception as e:
            return f'Error reading PDF: {e}'
    try:
        with open(file_path, 'r') as f:
            return '[CLAIM DOC]\n' + f.read()
    except FileNotFoundError:
        return f'File not found: {file_path}'

# ── TOOL 2: Extract Fields ─────────────────────────────────────
def extract_claim_fields(text: str) -> str:
    patterns = {
        'Claim ID':             r'Claim ID[:\s]+([A-Z0-9\-]+)',
        'Policy Number':        r'Policy Number[:\s]+([A-Z0-9\-]+)',
        'Claimant Name':        r'Name[:\s]+([A-Za-z\s]+?)(?:\n|Policy)',
        'Policy Type':          r'Policy Type[:\s]+(.+?)(?:\n|Coverage)',
        'Incident Type':        r'Type of Incident[:\s]+(.+?)(?:\n|Location)',
        'Date of Incident':     r'Date of Incident[:\s]+([0-9\-/]+)',
        'Total Claimed Amount': r'Total Claimed Amount[:\s]+\$?([\d,]+)',
        'Coverage Amount':      r'Coverage Amount[:\s]+\$?([\d,]+)',
        'Policy Start Date':    r'Policy Start Date[:\s]+([0-9\-/]+)',
    }
    out = '\nEXTRACTED CLAIM FIELDS:\n' + '-'*40 + '\n'
    for label, pat in patterns.items():
        m = re.search(pat, text, re.IGNORECASE)
        out += f'  {label}: {m.group(1).strip() if m else "NOT FOUND"}\n'
    out += f'  Has Police Report: {bool(re.search(r"police report.*attached", text, re.IGNORECASE))}\n'
    out += f'  Has Witness: {bool(re.search(r"witness", text, re.IGNORECASE) and not re.search(r"none provided", text, re.IGNORECASE))}\n'
    out += f'  Has Photos: {bool(re.search(r"photo", text, re.IGNORECASE))}\n'
    return out

# ── TOOL 3: Check Rules & Fraud Detection ─────────────────────
def check_policy_rules(claim_info: str) -> str:
    flags, passed, failed = [], [], []
    risk = 0
    text = claim_info.lower()

    # Rule 1: Claimed vs Coverage ratio
    cov = re.search(r'coverage amount[:\s]+\$?([\d,]+)', claim_info, re.IGNORECASE)
    clm = re.search(r'total claimed amount[:\s]+\$?([\d,]+)', claim_info, re.IGNORECASE)
    if cov and clm:
        coverage = int(cov.group(1).replace(',', ''))
        claimed  = int(clm.group(1).replace(',', ''))
        ratio = claimed / coverage
        if ratio > 0.9:
            flags.append(f'[HIGH] Claimed ${claimed:,} = {ratio*100:.0f}% of coverage')
            risk += 30
        elif ratio > 0.6:
            flags.append(f'[MEDIUM] Claimed ${claimed:,} = {ratio*100:.0f}% of coverage')
            risk += 10
        else:
            passed.append(f'Claim ratio {ratio*100:.0f}% - normal')

    # Rule 2: Policy age
    ps  = re.search(r'policy start date[:\s]+([0-9\-/]+)', claim_info, re.IGNORECASE)
    inc = re.search(r'date of incident[:\s]+([0-9\-/]+)', claim_info, re.IGNORECASE)
    if ps and inc:
        try:
            days = (datetime.strptime(inc.group(1), '%Y-%m-%d') -
                    datetime.strptime(ps.group(1),  '%Y-%m-%d')).days
            if days < 60:
                flags.append(f'[FRAUD RISK] Policy only {days} days old at incident')
                risk += 40
                failed.append(f'Policy too new ({days} days)')
            elif days < 180:
                flags.append(f'[MONITOR] Policy {days} days old')
                risk += 10
            else:
                passed.append(f'Policy age {days} days - normal')
        except:
            flags.append('[WARNING] Could not verify policy age')
            risk += 10

    # Rule 3: Documentation check
    has_police  = bool(re.search(r'police report.*attached', claim_info, re.IGNORECASE))
    has_photos  = bool(re.search(r'photo', claim_info, re.IGNORECASE))
    has_witness = bool(re.search(r'witness', claim_info, re.IGNORECASE) and
                       not re.search(r'none provided', claim_info, re.IGNORECASE))
    docs = sum([has_police, has_photos, has_witness])
    if docs >= 3:
        passed.append('Full documentation (police + photos + witness)')
    elif docs == 2:
        passed.append('Good documentation (2/3 items)')
        risk += 5
    elif docs == 1:
        flags.append('[WARNING] Only 1 of 3 documentation items present')
        risk += 20
        failed.append('Insufficient documentation')
    else:
        flags.append('[HIGH RISK] No supporting documentation')
        risk += 35
        failed.append('No documentation')

    # Rule 4: Theft with no forced entry
    if ('theft' in text or 'burglary' in text) and 'no signs of forced entry' in text:
        flags.append('[SUSPICIOUS] Theft with NO forced entry reported')
        risk += 35
        failed.append('Theft - no forced entry')

    # Rule 5: Cash loss
    if 'cash loss' in text:
        flags.append('[INFO] Cash loss claimed - unverifiable')
        risk += 10

    out  = '\nPOLICY COMPLIANCE ASSESSMENT:\n' + '-'*40 + '\n'
    out += f'  Risk Score: {risk}/100\n'
    out += f'  Risk Level: {"HIGH" if risk >= 50 else "MEDIUM" if risk >= 25 else "LOW"}\n\n'
    if passed:
        out += 'CHECKS PASSED:\n' + ''.join(f'  + {c}\n' for c in passed)
    if failed:
        out += '\nCHECKS FAILED:\n' + ''.join(f'  x {c}\n' for c in failed)
    if flags:
        out += '\nFLAGS:\n' + ''.join(f'  ! {f}\n' for f in flags)
    return out

# ── TOOL 4: Make Decision ──────────────────────────────────────
def make_claim_decision(assessment: str) -> str:
    m = re.search(r'Risk Score[:\s]+(\d+)', assessment, re.IGNORECASE)
    risk = int(m.group(1)) if m else 50

    if risk >= 60:
        decision   = 'ESCALATE TO SENIOR INVESTIGATOR'
        action     = 'Flag for manual investigation before processing'
        timeline   = '5-7 business days (pending investigation)'
        steps = ['Assign to Senior Claims Investigator',
                 'Request additional documentation',
                 'Conduct field inspection',
                 'Cross-check with fraud database',
                 'Schedule claimant interview']
    elif risk >= 30:
        decision   = 'CONDITIONAL APPROVAL - ADDITIONAL DOCS REQUIRED'
        action     = 'Request missing documents before final approval'
        timeline   = '3-5 business days after document receipt'
        steps = ['Send document request to claimant',
                 'Allow 14 days for submission',
                 'Assign to standard claims adjuster',
                 'Re-evaluate once docs received']
    else:
        decision   = 'APPROVED FOR PROCESSING'
        action     = 'Proceed with standard claims settlement'
        timeline   = '5-7 business days'
        steps = ['Initiate payment processing',
                 'Send approval to claimant',
                 'Update claims system',
                 'Close claim on payment confirmation']

    out  = '\nFINAL CLAIM DECISION:\n' + '='*45 + '\n'
    out += f'  Decision:   {decision}\n'
    out += f'  Action:     {action}\n'
    out += f'  Timeline:   {timeline}\n'
    out += f'  Risk Score: {risk}/100\n'
    out += '\nNEXT STEPS:\n'
    for i, s in enumerate(steps, 1):
        out += f'  {i}. {s}\n'
    out += '='*45
    return out

# ── TOOL 5: Generate Report ────────────────────────────────────
def generate_claim_report(summary: str) -> str:
    ts       = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    filename = f'reports/report_{datetime.now().strftime("%Y%m%d_%H%M%S")}.txt'
    report   = f"""
==========================================================
       CHUBB INSURANCE - OFFICIAL CLAIMS REPORT
       Generated: {ts}
==========================================================
{summary}
----------------------------------------------------------
  Generated by: Chubb AI Claims Processing Agent v1.0
  Contact: claims@chubb.com | www.chubb.com
----------------------------------------------------------
"""
    os.makedirs('reports', exist_ok=True)
    with open(filename, 'w') as f:
        f.write(report)
    return f'Report saved: {filename}\n{report}'

print("All 5 tools defined:")
print("  Tool 1: read_claim_document   - reads TXT/PDF")
print("  Tool 2: extract_claim_fields  - extracts structured data")
print("  Tool 3: check_policy_rules    - fraud detection & risk score")
print("  Tool 4: make_claim_decision   - APPROVE / ESCALATE / REJECT")
print("  Tool 5: generate_claim_report - saves formal report")

All 5 tools defined:
  Tool 1: read_claim_document   - reads TXT/PDF
  Tool 2: extract_claim_fields  - extracts structured data
  Tool 3: check_policy_rules    - fraud detection & risk score
  Tool 4: make_claim_decision   - APPROVE / ESCALATE / REJECT
  Tool 5: generate_claim_report - saves formal report


In [5]:
import os
from langchain_groq import ChatGroq
from langchain.agents import create_react_agent
from langchain.agents.agent import AgentExecutor   # ✅ FIXED
from langchain.tools import Tool
from langchain.prompts import PromptTemplate

# Register tools
tools = [
    Tool(name='ReadClaimDocument',
         func=read_claim_document,
         description='Read a claim file. Input: file path like sample_claims/claim_001_fire.txt. Returns full document text.'),
    Tool(name='ExtractClaimFields',
         func=extract_claim_fields,
         description='Extract structured fields from claim text. Input: raw document text. Returns claim ID, amounts, dates etc.'),
    Tool(name='CheckPolicyRules',
         func=check_policy_rules,
         description='Check fraud indicators and underwriting rules. Input: extracted claim fields text. Returns risk score and flags.'),
    Tool(name='MakeClaimDecision',
         func=make_claim_decision,
         description='Make final decision. Input: assessment text with risk score. Returns APPROVE / ESCALATE / REJECT with next steps.'),
    Tool(name='GenerateClaimReport',
         func=generate_claim_report,
         description='Save a formal report. Input: full summary of findings. Returns saved report.'),
]

# Prompt
PROMPT = PromptTemplate.from_template("""
You are a Chubb Insurance Claims Processing Agent.
Always follow this exact order: Read -> Extract -> Check Rules -> Decide -> Report.
Be professional and justify every decision clearly.

Tools available:
{tools}

Use EXACTLY this format:

Question: the task
Thought: what I need to do next
Action: tool name from [{tool_names}]
Action Input: input for the tool
Observation: result from tool
... (repeat until done)
Thought: I have everything I need
Final Answer: clear professional summary of the claim decision

Begin!

Question: {input}
Thought:{agent_scratchpad}
""")

# ✅ FIXED LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=os.environ["GROQ_API_KEY"],
    temperature=0.1,
)

# Build agent
agent = create_react_agent(llm, tools, PROMPT)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    max_iterations=10,
    handle_parsing_errors=True,
)

print("✅ Agent built successfully with Groq!")

✅ Agent built successfully with Groq!


In [6]:
def process_claim(file_path: str):
    print('\n' + '='*60)
    print(f'  CHUBB CLAIMS AGENT - Processing: {file_path}')
    print('='*60)

    print('\n[Step 1] Reading document...')
    doc = read_claim_document(file_path)

    print('[Step 2] Extracting fields...')
    extracted = extract_claim_fields(doc)
    print(extracted)

    print('[Step 3] Checking rules & fraud indicators...')
    assessment = check_policy_rules(extracted + '\n' + doc)
    print(assessment)

    print('[Step 4] Making decision...')
    decision = make_claim_decision(assessment)
    print(decision)

    print('[Step 5] Saving report...')
    report = generate_claim_report(f'File: {file_path}\n{extracted}\n{assessment}\n{decision}')
    print(report.split('\n')[0])

    print('\n' + '='*60)
    print('  DONE')
    print('='*60)

# Run all 3 claims
process_claim('sample_claims/claim_001_fire.txt')


  CHUBB CLAIMS AGENT - Processing: sample_claims/claim_001_fire.txt

[Step 1] Reading document...
[Step 2] Extracting fields...

EXTRACTED CLAIM FIELDS:
----------------------------------------
  Claim ID: CLM-2024-00123
  Policy Number: POL-789456
  Claimant Name: Rajesh Kumar
  Policy Type: Commercial Property Insurance
  Incident Type: Fire Damage
  Date of Incident: 2024-03-10
  Total Claimed Amount: 235,000
  Coverage Amount: 500,000
  Policy Start Date: 2023-01-01
  Has Police Report: True
  Has Witness: True
  Has Photos: True

[Step 3] Checking rules & fraud indicators...

POLICY COMPLIANCE ASSESSMENT:
----------------------------------------
  Risk Score: 0/100
  Risk Level: LOW

CHECKS PASSED:
  + Claim ratio 47% - normal
  + Policy age 434 days - normal
  + Full documentation (police + photos + witness)

[Step 4] Making decision...

FINAL CLAIM DECISION:
  Decision:   APPROVED FOR PROCESSING
  Action:     Proceed with standard claims settlement
  Timeline:   5-7 business da

In [7]:
process_claim('sample_claims/claim_002_theft.txt')


  CHUBB CLAIMS AGENT - Processing: sample_claims/claim_002_theft.txt

[Step 1] Reading document...
[Step 2] Extracting fields...

EXTRACTED CLAIM FIELDS:
----------------------------------------
  Claim ID: CLM-2024-00456
  Policy Number: POL-112233
  Claimant Name: Vikram Singh
  Policy Type: Personal Property Insurance
  Incident Type: Theft
  Date of Incident: 2024-03-18
  Total Claimed Amount: 48,000
  Coverage Amount: 50,000
  Policy Start Date: 2024-02-01
  Has Police Report: True
  Has Witness: False
  Has Photos: True

[Step 3] Checking rules & fraud indicators...

POLICY COMPLIANCE ASSESSMENT:
----------------------------------------
  Risk Score: 120/100
  Risk Level: HIGH

CHECKS PASSED:
  + Good documentation (2/3 items)

CHECKS FAILED:
  x Policy too new (46 days)
  x Theft - no forced entry

FLAGS:
  ! [HIGH] Claimed $48,000 = 96% of coverage
  ! [FRAUD RISK] Policy only 46 days old at incident
  ! [SUSPICIOUS] Theft with NO forced entry reported
  ! [INFO] Cash loss cla

In [8]:
process_claim('sample_claims/claim_003_water.txt')


  CHUBB CLAIMS AGENT - Processing: sample_claims/claim_003_water.txt

[Step 1] Reading document...
[Step 2] Extracting fields...

EXTRACTED CLAIM FIELDS:
----------------------------------------
  Claim ID: CLM-2024-00789
  Policy Number: POL-334455
  Claimant Name: Priya Sharma
  Policy Type: Commercial Property Insurance
  Incident Type: Water Damage
  Date of Incident: 2024-03-28
  Total Claimed Amount: 150,000
  Coverage Amount: 200,000
  Policy Start Date: 2022-06-01
  Has Police Report: False
  Has Witness: True
  Has Photos: True

[Step 3] Checking rules & fraud indicators...

POLICY COMPLIANCE ASSESSMENT:
----------------------------------------
  Risk Score: 15/100
  Risk Level: LOW

CHECKS PASSED:
  + Policy age 666 days - normal
  + Good documentation (2/3 items)

FLAGS:
  ! [MEDIUM] Claimed $150,000 = 75% of coverage

[Step 4] Making decision...

FINAL CLAIM DECISION:
  Decision:   APPROVED FOR PROCESSING
  Action:     Proceed with standard claims settlement
  Timeline:   

In [9]:
# Watch the agent THINK step by step

result = agent_executor.invoke({
    "input": """
    Process the claim at: sample_claims/claim_001_fire.txt
    Read it, extract fields, check rules, make decision, generate report.
    """
})

print("\n" + "="*60)
print("AGENT FINAL ANSWER:")
print("="*60)
print(result["output"])



> Entering new AgentExecutor chain...
To process the claim, I need to start by reading the claim document to obtain the necessary information. 

Action: ReadClaimDocument
Action Input: sample_claims/claim_001_fire.txt[CLAIM DOC]

CHUBB INSURANCE - CLAIM DOCUMENT
Claim ID: CLM-2024-00123
Date of Filing: 2024-03-15
Policy Number: POL-789456

POLICYHOLDER INFORMATION
Name: Rajesh Kumar
Policy Type: Commercial Property Insurance
Coverage Amount: $500,000
Premium Status: Active
Policy Start Date: 2023-01-01
Policy End Date: 2025-01-01

INCIDENT DETAILS
Date of Incident: 2024-03-10
Type of Incident: Fire Damage
Location: Warehouse Unit 4B, Industrial Area, Bangalore
Description: Fire broke out due to electrical short circuit.
             60% of stored goods were damaged or destroyed.

DAMAGE ASSESSMENT
Property Damage: $120,000
Inventory Loss: $85,000
Business Interruption Loss: $30,000
Total Claimed Amount: $235,000

SUPPORTING DOCUMENTS
- Fire Department Report: Attached (FD-2024-0315)


In [10]:
import glob

claims = sorted(glob.glob('sample_claims/*.txt'))
results = []

for f in claims:
    doc        = read_claim_document(f)
    extracted  = extract_claim_fields(doc)
    assessment = check_policy_rules(extracted + '\n' + doc)
    decision   = make_claim_decision(assessment)

    score = int(re.search(r'Risk Score[:\s]+(\d+)', assessment).group(1))
    dec   = re.search(r'Decision:\s+(.+)', decision)
    dec   = dec.group(1).strip() if dec else 'N/A'
    amt   = re.search(r'Total Claimed Amount[:\s]+\$?([\d,]+)', extracted)
    amt   = f"${amt.group(1)}" if amt else 'N/A'

    results.append({'file': f.split('/')[-1], 'amount': amt, 'score': score, 'decision': dec})

print('='*75)
print(f'  {"FILE":<35} {"AMOUNT":<12} {"RISK":<6} DECISION')
print('='*75)
for r in results:
    icon = 'GREEN' if r['score'] < 30 else 'YELLOW' if r['score'] < 60 else 'RED'
    print(f'  {r["file"]:<35} {r["amount"]:<12} {r["score"]:<6} [{icon}] {r["decision"][:30]}')
print('='*75)
print(f'\n  Total: {len(results)} | '
      f'Approve: {sum(1 for r in results if r["score"]<30)} | '
      f'Escalate: {sum(1 for r in results if 30<=r["score"]<60)} | '
      f'Reject: {sum(1 for r in results if r["score"]>=60)}')

  FILE                                AMOUNT       RISK   DECISION
  claim_001_fire.txt                  $235,000     0      [GREEN] APPROVED FOR PROCESSING
  claim_002_theft.txt                 $48,000      120    [RED] ESCALATE TO SENIOR INVESTIGATO
  claim_003_water.txt                 $150,000     15     [GREEN] APPROVED FOR PROCESSING

  Total: 3 | Approve: 2 | Escalate: 0 | Reject: 1


In [11]:
from google.colab import files
import tempfile

print("Upload your own claim file (TXT or PDF):")
uploaded = files.upload()

for filename, content in uploaded.items():
    path = f'sample_claims/{filename}'
    with open(path, 'wb') as f:
        f.write(content)
    print(f'\nProcessing: {filename}')
    process_claim(path)

Upload your own claim file (TXT or PDF):


Saving claim_001.txt to claim_001.txt

Processing: claim_001.txt

  CHUBB CLAIMS AGENT - Processing: sample_claims/claim_001.txt

[Step 1] Reading document...
[Step 2] Extracting fields...

EXTRACTED CLAIM FIELDS:
----------------------------------------
  Claim ID: CLM-2024-00123
  Policy Number: POL-789456
  Claimant Name: Rajesh Kumar
  Policy Type: Commercial Property Insurance
  Incident Type: Fire Damage
  Date of Incident: 2024-03-10
  Total Claimed Amount: 235,000
  Coverage Amount: 500,000
  Policy Start Date: 2023-01-01
  Has Police Report: True
  Has Witness: True
  Has Photos: True

[Step 3] Checking rules & fraud indicators...

POLICY COMPLIANCE ASSESSMENT:
----------------------------------------
  Risk Score: 0/100
  Risk Level: LOW

CHECKS PASSED:
  + Claim ratio 47% - normal
  + Policy age 434 days - normal
  + Full documentation (police + photos + witness)

[Step 4] Making decision...

FINAL CLAIM DECISION:
  Decision:   APPROVED FOR PROCESSING
  Action:     Proceed w

In [12]:
from google.colab import files
import glob

for report in glob.glob('reports/*.txt'):
    print(f'Downloading: {report}')
    files.download(report)

Downloading: reports/report_20260408_121346.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: reports/report_20260408_121033.txt


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloading: reports/report_20260408_121037.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>